In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os

# 1. Load features
df = pd.read_parquet('../data/nifty_features.parquet')
print(f"Loaded {len(df)} rows")

features = ['Log_Return_1d', 'Log_Return_5d', 'RSI', 'Volatility']

# 2. Custom Asymmetric Volatility-Penalized Loss (AVP-Loss)
# Loss: (1 + lambda * sigma) * (y - y_hat)^2  if y_hat > y  (over-prediction)
#       (y - y_hat)^2                          if y_hat <= y (under-prediction)
# Gradient: g = -2 * (1 + lambda*sigma) * (y - y_hat)  if over
#           g = -2 * (y - y_hat)                       if under
# Hessian:  h = 2 * (1 + lambda*sigma)                 if over
#           h = 2                                      if under

def avp_objective(preds, dtrain):
    y = dtrain.get_label()
    sigma = dtrain.get_weight()
    lam = 2.5
    err = y - preds
    over = preds > y
    mult = np.where(over, 1.0 + lam * sigma, 1.0)
    grad = -2.0 * mult * err
    hess = 2.0 * mult
    return grad, hess

# 3. Walk-forward with custom loss
train_window = 1260
test_window = 252

preds_custom = []
actuals_custom = []
dates_custom = []

print("Running walk-forward with AVP-Loss...")

for i in range(train_window, len(df) - test_window, test_window):
    train = df.iloc[:i]
    test = df.iloc[i:i+test_window]
    
    X_train, y_train = train[features], train['Target']
    X_test, y_test = test[features], test['Target']
    
    dtrain = xgb.DMatrix(X_train, label=y_train, weight=train['Volatility'])
    dtest = xgb.DMatrix(X_test)
    
    params = {'max_depth': 4, 'learning_rate': 0.05}
    bst = xgb.train(params, dtrain, num_boost_round=100, obj=avp_objective)
    pred = bst.predict(dtest)
    
    preds_custom.extend(pred)
    actuals_custom.extend(y_test.values)
    dates_custom.extend(test.index)
    print(f"  Trained up to {train.index[-1].date()}, tested {len(pred)} days")

# 4. Save
os.makedirs('../reports', exist_ok=True)
results_custom = pd.DataFrame({
    'Date': dates_custom,
    'Actual': actuals_custom,
    'Pred': preds_custom
})
results_custom.to_csv('../reports/custom_results.csv', index=False)

rmse_custom = np.sqrt(mean_squared_error(results_custom['Actual'], results_custom['Pred']))
mae_custom = mean_absolute_error(results_custom['Actual'], results_custom['Pred'])
direction_acc_custom = ((results_custom['Pred'] > 0) == (results_custom['Actual'] > 0)).mean()

print(f"\n{'='*50}")
print(f"CUSTOM AVP-LOSS RESULTS")
print(f"{'='*50}")
print(f"RMSE:                 {rmse_custom:.6f}")
print(f"MAE:                  {mae_custom:.6f}")
print(f"Directional Accuracy: {direction_acc_custom*100:.2f}%")
print(f"Total predictions:    {len(results_custom)}")
print(f"{'='*50}")
print("\nCustom results saved to reports/custom_results.csv")

Loaded 2439 rows
Running walk-forward with AVP-Loss...
  Trained up to 2019-03-18, tested 252 days
  Trained up to 2020-03-31, tested 252 days
  Trained up to 2021-04-07, tested 252 days
  Trained up to 2022-04-12, tested 252 days

CUSTOM AVP-LOSS RESULTS
RMSE:                 0.013989
MAE:                  0.008964
Directional Accuracy: 53.08%
Total predictions:    1008

Custom results saved to reports/custom_results.csv
